
# Ingesta SharePoint → Bronze

## Propósito

Extraer los datos desde SharePoint correspondientes al corte previamente validado por `prepare_pipeline_control` y almacenarlos en la capa Bronze en formato Delta, conservando la trazabilidad de la ejecución, del archivo de origen y del proceso ETL.

## Flujo

1. Obtener `fecha_inicio`, `fecha_fin` e `id_ejecucion` mediante Task Values.

2. Generar el `log_id` correspondiente a la ejecución de este notebook.

3. Validar los parámetros recibidos.

4. Registrar el inicio de la ejecución en `etl_log` con estado `EN_PROCESO`.

5. Autenticar el acceso a SharePoint mediante Microsoft Graph.

6. Validar la existencia y el acceso al archivo de origen.

7. Descargar la fuente de datos al área RAW.

8. Leer y procesar los datos con PySpark.

9. Aplicar el corte de extracción según `fecha_inicio` y `fecha_fin`.

10. Agregar los metadatos de trazabilidad.

11. Normalizar los nombres de las columnas.

12. Validar que el corte no haya sido cargado previamente en Bronze.

13. Escribir los datos en Bronze en formato Delta cuando el corte no exista.

14. Validar los registros almacenados en Bronze.

15. Finalizar el registro en `etl_log` con estado `EXITOSO` y almacenar la cantidad de filas insertadas.

## Origen

SharePoint.

**Archivo de origen:**

`DataCoSupplyChainDataset.csv`

## Área RAW

El archivo descargado temporalmente se almacena en el Volume RAW:

`/Volumes/workspace/raw/sharepoint`

## Destino

Capa Bronze.

**Catálogo:**

`brz_dataco`

**Esquema:**

`sharepoint`

**Tabla:**

`brz_dataco.sharepoint.raw_dataco_supply_chain`

## Parámetros recibidos

* `fecha_inicio`
* `fecha_fin`
* `id_ejecucion`

Estos valores son publicados por `prepare_pipeline_control` mediante Task Values.

## Identificadores de ejecución

El proceso utiliza dos identificadores:

* `id_ejecucion`: identifica la ejecución completa del Workflow.
* `log_id`: identifica específicamente la ejecución de este notebook en `etl_log`.

## Trazabilidad generada

La carga Bronze conserva los siguientes metadatos:

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`
* `origen`
* `archivo_origen`

## Control de ejecución

El notebook registra su ejecución en:

`workspace.metadata.etl_log`

El estado del proceso sigue el siguiente flujo:

`EN_PROCESO → EXITOSO`

En caso de error, el registro será actualizado posteriormente por el mecanismo de manejo de errores del Workflow.

La columna `filas_procesadas` registra la cantidad de registros realmente insertados en Bronze.

Si el corte ya existe, el proceso no vuelve a insertar los registros y `filas_procesadas` queda en `0`.

## Dependencias

Este notebook debe ejecutarse como una **Task dentro de un Databricks Workflow**, después de `prepare_pipeline_control`, para poder consumir los valores publicados mediante Task Values.

## Resultado esperado

Una carga Bronze correspondiente únicamente al rango de fechas recibido desde `prepare_pipeline_control`, almacenada en formato Delta y con la trazabilidad completa de la ejecución y del archivo de origen.

El proceso debe ser controlado para evitar la duplicidad de un mismo corte de extracción.



In [0]:
import uuid

# ============================================================
# 1. GENERACIÓN DEL LOG ID
# ============================================================
# Genera una clave primaria fija en Python independiente del entorno Spark.
log_id = str(uuid.uuid4())

print(f"✓ log_id generado: {log_id}")

In [0]:
# ============================================================
# 2. OBTENCIÓN DE PARÁMETROS DE EXTRACCIÓN
# ============================================================
fecha_inicio = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="fecha_inicio",
    debugValue=""
)

fecha_fin = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="fecha_fin",
    debugValue=""
)

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

print(f"✓ Fecha inicio: {fecha_inicio}")
print(f"✓ Fecha fin:    {fecha_fin}")
print(f"✓ ID ejecución: {id_ejecucion}")

In [0]:
import pyspark.sql.functions as F

# ============================================================
# 3. INICIO DEL REGISTRO DE EJECUCIÓN
# ============================================================
TABLA_ETL_LOG = "metadata_dataco.supply_chain.hd_etl_log"
TASK_NAME = "ingest_sharepoint_bronze"

df_log_inicio = spark.createDataFrame(
    [
        (
            log_id,
            id_ejecucion,
            TASK_NAME,
            "bronze",
            None,
            None,
            None,
            "EN_PROCESO",
            None
        )
    ],
    """
    log_id STRING,
    id_ejecucion STRING,
    notebook STRING,
    capa STRING,
    fecha_inicio TIMESTAMP,
    fecha_fin TIMESTAMP,
    filas_procesadas BIGINT,
    estado STRING,
    mensaje_error STRING
    """
).withColumn("fecha_inicio", F.current_timestamp())

df_log_inicio.write.format("delta").mode("append").saveAsTable(TABLA_ETL_LOG)

print(f"✓ Registro EN_PROCESO insertado en {TABLA_ETL_LOG}")

In [0]:
# ============================================================
# 4. PUBLICACIÓN DEL LOG ID
# ============================================================
# Se publica inmediatamente después de guardar la fila en la tabla Delta.
dbutils.jobs.taskValues.set(
    key="log_id",
    value=log_id
)

print(f"✓ log_id publicado correctamente en taskValues: {log_id}")

In [0]:
# ============================================================
# VALIDACIÓN DEL RANGO DE FECHAS
# ============================================================

from datetime import datetime

# Convierte las fechas recibidas desde el Task anterior
# al tipo date para realizar las validaciones.
fecha_inicio_dt = datetime.strptime(
    fecha_inicio,
    "%Y-%m-%d"
).date()

fecha_fin_dt = datetime.strptime(
    fecha_fin,
    "%Y-%m-%d"
).date()

# Valida que la fecha inicial no sea posterior
# a la fecha final.
if fecha_inicio_dt > fecha_fin_dt:
    raise ValueError(
        f"Rango de fechas inválido: "
        f"fecha_inicio={fecha_inicio}, "
        f"fecha_fin={fecha_fin}"
    )

print("✓ Parámetros de extracción válidos.")
print(f"✓ Rango: {fecha_inicio} → {fecha_fin}")

In [0]:
# ============================================================
# OBTENCIÓN DE PARÁMETROS DE SHAREPOINT
# ============================================================

import re

# Define los parámetros utilizados para autenticarse
# contra Microsoft Graph.
dbutils.widgets.text("SP_TENANT_ID", "")
dbutils.widgets.text("SP_CLIENT_ID", "")


tenant_id = dbutils.widgets.get("SP_TENANT_ID")
client_id = dbutils.widgets.get("SP_CLIENT_ID")

# El Client Secret se lee directo del secret scope (no por Job Parameter)
client_secret = dbutils.secrets.get(scope="sharepoint", key="sp_client_secret")

# Diagnóstico sin exponer el secreto
es_guid = bool(re.fullmatch(
    r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}",
    client_secret.strip()
))


print(f"Tenant ID recibido: {tenant_id}")
print(f"Client ID recibido: {client_id}")
print(f"Client Secret: longitud={len(client_secret)} | parece GUID (Secret ID)={es_guid}")

if es_guid:
    raise ValueError(
        "El secreto guardado parece un Secret ID (GUID), no el Value. "
        "Copia el campo 'Value' del secreto en Azure y vuelve a guardarlo en el scope."
    )

In [0]:
# ============================================================
# AUTENTICACIÓN MICROSOFT GRAPH
# ============================================================

import requests

token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"

token_data = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": "https://graph.microsoft.com/.default"
}

response = requests.post(token_url, data=token_data)

if response.status_code != 200:
    raise RuntimeError(
        f"Error al obtener token de Microsoft Graph: "
        f"{response.status_code} - {response.text}"
    )

access_token = response.json()["access_token"]

print("Token de Microsoft Graph obtenido correctamente.")

In [0]:
# ============================================================
# OBTENER SITE ID DE SHAREPOINT
# ============================================================

headers = {
    "Authorization": f"Bearer {access_token}"
}

site_url = (
    "https://graph.microsoft.com/v1.0/"
    "sites/technologicalsolutionsbpo.sharepoint.com:"
    "/sites/Storage-OnPremise2"
)

response = requests.get(site_url, headers=headers)

if response.status_code != 200:
    raise RuntimeError(
        f"Error al obtener SharePoint Site ID: "
        f"{response.status_code} - {response.text}"
    )

site_data = response.json()

site_id = site_data["id"]
site_name = site_data.get("displayName", "Sin nombre")

print(f"SharePoint encontrado: {site_name}")
print("Site ID obtenido correctamente.")

In [0]:
# ============================================================
# VALIDAR ACCESO AL ARCHIVO DE SHAREPOINT
# ============================================================

NOMBRE_ARCHIVO = "DataCoSupplyChainDataset.csv"
RUTA_CARPETA = "3. Reporte Performance Logistico DataCo"

ruta_archivo = f"{RUTA_CARPETA}/{NOMBRE_ARCHIVO}"

file_url = (
    f"https://graph.microsoft.com/v1.0/"
    f"sites/{site_id}/drive/root:/{ruta_archivo}"
)

response = requests.get(file_url, headers=headers)

if response.status_code != 200:
    raise RuntimeError(
        f"Error al localizar el archivo: "
        f"{response.status_code} - {response.text}"
    )

file_data = response.json()

print(f"Archivo encontrado: {file_data.get('name')}")
print("Acceso al archivo validado correctamente.")

In [0]:
%sql

-- ============================================================
-- CREACIÓN DE CATÁLOGO Y ESQUEMA BRONZE
-- ============================================================

CREATE CATALOG IF NOT EXISTS brz_dataco;

CREATE SCHEMA IF NOT EXISTS brz_dataco.sharepoint;

In [0]:
%sql

-- ============================================================
-- CREACIÓN DEL VOLUME DE ARCHIVOS DE ORIGEN
-- ============================================================

-- Asegura la existencia del catálogo y del esquema del sistema fuente
CREATE CATALOG IF NOT EXISTS brz_dataco;
CREATE SCHEMA IF NOT EXISTS brz_dataco.sharepoint;

-- Crea el Volume administrado para los archivos de origen de SharePoint
CREATE VOLUME IF NOT EXISTS brz_dataco.sharepoint.archivos;

In [0]:
# ============================================================
# DESCARGAR CSV COMPLETO A RAW
# ============================================================

import requests

# Ruta estandarizada de Unity Catalog Volume (3 niveles: /Volumes/catalogo/esquema/volumen)
RUTA_RAW = "/Volumes/brz_dataco/sharepoint/archivos"
RUTA_RAW_ARCHIVO = f"{RUTA_RAW}/{NOMBRE_ARCHIVO}"

file_content_url = (
    f"https://graph.microsoft.com/v1.0/"
    f"sites/{site_id}/drive/root:/{ruta_archivo}:/content"
)

response = requests.get(
    file_content_url,
    headers=headers,
    stream=True
)

if response.status_code != 200:
    raise RuntimeError(
        f"Error al descargar el archivo: "
        f"{response.status_code} - {response.text}"
    )

with open(RUTA_RAW_ARCHIVO, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("✓ Archivo descargado correctamente a Raw Volume:")
print(RUTA_RAW_ARCHIVO)

In [0]:
# ============================================================
# VALIDACIÓN DEL CONTENIDO DEL VOLUME RAW
# ============================================================

display(
    dbutils.fs.ls(
        "/Volumes/brz_dataco/sharepoint/archivos"
    )
)

In [0]:
# ============================================================
# CONFIGURACIÓN DE RUTA RAW
# ============================================================

# Ruta estandarizada de Unity Catalog Volume (3 niveles: /Volumes/catalogo/esquema/volumen)
RUTA_RAW = "/Volumes/brz_dataco/sharepoint/archivos"
RUTA_RAW_ARCHIVO = f"{RUTA_RAW}/{NOMBRE_ARCHIVO}"

print(f"Ruta RAW: {RUTA_RAW_ARCHIVO}")

In [0]:
# ============================================================
# LECTURA DEL CSV DESDE RAW
# ============================================================

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .option("quote", '"')
    .option("escape", '"')
    .csv(RUTA_RAW_ARCHIVO)
)

print(f"Cantidad de columnas: {len(df_raw.columns)}")
print("Columnas:")
print(df_raw.columns)

display(df_raw.limit(5))

In [0]:
df_raw.printSchema()

In [0]:
# ============================================================
# CONVERSIÓN Y FILTRO DE FECHA DE EXTRACCIÓN
# ============================================================

from pyspark.sql import functions as F

COLUMNA_FECHA = "order date (DateOrders)"

df_bronze = (
    df_raw
    .withColumn(
        "order_date",
        F.to_timestamp(
            F.col(COLUMNA_FECHA),
            "M/d/yyyy H:mm"
        ).cast("date")
    )
    .filter(
        (F.col("order_date") >= F.to_date(F.lit(fecha_inicio))) &
        (F.col("order_date") <= F.to_date(F.lit(fecha_fin)))
    )
)

filas_raw = df_raw.count()
filas_procesadas = df_bronze.count()

print(f"✓ Registros RAW:       {filas_raw}")
print(f"✓ Registros procesados: {filas_procesadas}")
print(f"✓ Rango aplicado:      {fecha_inicio} → {fecha_fin}")

In [0]:
# ============================================================
# AGREGAR METADATOS DEL CORTE Y TRAZABILIDAD
# ============================================================

df_bronze = (
    df_bronze
    .withColumn("fecha_inicio_corte", F.to_date(F.lit(fecha_inicio)))
    .withColumn("fecha_fin_corte", F.to_date(F.lit(fecha_fin)))
    .withColumn("fecha_carga", F.current_timestamp())
    .withColumn("id_ejecucion", F.lit(id_ejecucion))
    .withColumn("origen", F.lit("sharepoint"))
    .withColumn("archivo_origen", F.lit(NOMBRE_ARCHIVO))
)

In [0]:
# ============================================================
# VALIDACIÓN DEL ESQUEMA DE TRAZABILIDAD
# ============================================================

df_bronze.select(
    "fecha_inicio_corte",
    "fecha_fin_corte",
    "fecha_carga",
    "id_ejecucion",
    "origen",
    "archivo_origen"
).printSchema()

In [0]:
# ============================================================
# NORMALIZACIÓN DE NOMBRES DE COLUMNAS PARA BRONZE
# ============================================================

import re

def normalizar_nombre_columna(nombre):
    nombre = nombre.strip().lower()
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    nombre = re.sub(r"_+", "_", nombre)
    nombre = nombre.strip("_")
    return nombre


df_bronze = df_bronze.toDF(
    *[normalizar_nombre_columna(c) for c in df_bronze.columns]
)

print("Nombres de columnas normalizados correctamente.")
print(df_bronze.columns)

In [0]:
# ============================================================
# VALIDACIÓN DE REGISTROS PROCESADOS
# ============================================================

display(
    df_bronze
    .select(
        "order_date_dateorders",
        "order_date",
        "order_id"
    )
    .orderBy("order_date")
    .limit(20)
)

In [0]:
# ============================================================
# CONTROL DE EXISTENCIA DEL CORTE EN BRONZE
# ============================================================

# Definición con nomenclatura estándar de 3 niveles en Unity Catalog
TABLA_BRONZE = "brz_dataco.sharepoint.raw_supply_chain"
tabla_existe = spark.catalog.tableExists(TABLA_BRONZE)

if tabla_existe:

    df_corte_existente = (
        spark.table(TABLA_BRONZE)
        .filter(
            (F.col("fecha_inicio_corte") == F.to_date(F.lit(fecha_inicio))) &
            (F.col("fecha_fin_corte") == F.to_date(F.lit(fecha_fin)))
        )
        .limit(1)
    )

    corte_existe = df_corte_existente.count() > 0

else:
    corte_existe = False

# ============================================================
# VALIDACIÓN DEL ESQUEMA ACTUAL DE BRONZE
# ============================================================

if tabla_existe:
    spark.table(TABLA_BRONZE).printSchema()

# ============================================================
# ESCRITURA DEL CORTE EN BRONZE
# ============================================================

if corte_existe:

    print(
        f"El corte {fecha_inicio} → {fecha_fin} "
        f"ya existe en Bronze. No se insertarán registros."
    )

else:

    (
        df_bronze
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_BRONZE)
    )

    print(
        f"Corte {fecha_inicio} → {fecha_fin} "
        f"insertado correctamente en Bronze ({TABLA_BRONZE})."
    )

    print(f"Registros insertados: {df_bronze.count()}")

In [0]:
# ============================================================
# VALIDACIÓN DE REGISTROS EN BRONZE
# ============================================================

display(
    spark.table(TABLA_BRONZE)
    .select(
        "fecha_inicio_corte",
        "fecha_fin_corte",
        "order_date",
        "order_id"
    )
    .orderBy("order_date")
    .limit(20)
)

In [0]:
# ============================================================
# FINALIZACIÓN EXITOSA DE LA EJECUCIÓN
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Obtiene la tabla Delta donde se registra
# la trazabilidad de las ejecuciones.
delta_log = DeltaTable.forName(
    spark,
    TABLA_ETL_LOG
)

# Actualiza únicamente el registro correspondiente
# a esta ejecución del notebook.
delta_log.update(
    condition=F.col("log_id") == log_id,
    set={
        "fecha_fin": F.current_timestamp(),
        "estado": F.lit("EXITOSO"),
        "filas_procesadas": F.lit(filas_procesadas),
        "mensaje_error": F.lit(None)
    }
)

print("✓ Ejecución finalizada correctamente.")
print(f"✓ Task: ingest_sharepoint_bronze")
print(f"✓ Log ID: {log_id}")
print(f"✓ Estado: EXITOSO")
print(f"✓ Filas procesadas: {filas_procesadas}")